In [41]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, BitsAndBytesConfig
from datasets import load_dataset
import swanlab
import json
import os
import pandas as pd
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset


In [ ]:
# 检查 GPU 是否可用
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct", 
    device_map="cuda:0",
    dtype=torch.float16
)
print(f"Model device: {next(model.parameters()).device}")

# 使用流式模式加载数据
#train_data = load_dataset('json', data_files='training_data.jsonl', split='train', streaming=True)
#print(f"Dataset loaded in streaming mode")

Prompt_dict = {"prompt_no_input": """<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n<|im_end|>\n<|im_start|>assistant\n""",
    "prompt_input": """<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{input}<|im_end|>\n<|im_start|>assistant\n"""
    }
print(f"Prompt dict type: {type(Prompt_dict)}")

GPU Available: True
GPU Count: 1
Current GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Model device: cuda:0
Prompt dict type: <class 'dict'>


: 

In [ ]:
def json_to_dict(inp, out) -> str:
    prompt = Prompt_dict["prompt_input"].format(instruction="请根据现象分析", input=inp, output=out)
    print(prompt)
    return prompt


def process_sample(sample, tokenizer, prompt_dict):
    """处理单个样本的函数"""
    instruction = sample.get("instruction", "")
    input_text = sample.get("input", "")
    output_text = sample.get("response", "")
    
    # 构建 prompt
    if input_text:
        prompt = prompt_dict["prompt_input"].format(
            instruction=instruction, 
            input=input_text
        )
    else:
        prompt = prompt_dict["prompt_no_input"].format(
            instruction=instruction
        )
    
    full_text = prompt + output_text
    
    # tokenize
    encodings = tokenizer(
        full_text,
        truncation=True,
        max_length=tokenizer.model_max_length,
        padding="longest",
        return_tensors="pt"
    )
    
    # 获取input_ids
    input_ids = encodings["input_ids"].squeeze()
    
    # 创建labels（初始与input_ids相同）
    labels = input_ids.clone()
    
    # ========== 关键：添加loss掩码 ==========
    # 计算prompt的长度（需要忽略的部分）
    prompt_encoding = tokenizer(
        prompt,
        truncation=True,
        max_length=1024,
        return_tensors="pt"
    )
    prompt_length = len(prompt_encoding["input_ids"][0])
    
    # 将prompt部分的labels设置为-100（忽略loss）
    labels[:prompt_length] = -100
    
    return {
        "input_ids": input_ids,
        "attention_mask": encodings["attention_mask"].squeeze(),
        "labels": labels
    }

In [38]:
if __name__ == "__main__":
    import torch
    from torch.nn.utils.rnn import pad_sequence
    from functools import partial
    
    # 定义数据整理函数
    def collate_fn(batch):
        """整理批处理数据"""
        input_ids = []
        attention_mask = []
        labels = []
        
        for item in batch:
            iid = item["input_ids"]
            if isinstance(iid, torch.Tensor):
                input_ids.append(iid)
            else:
                input_ids.append(torch.tensor(iid))
            
            am = item["attention_mask"]
            if isinstance(am, torch.Tensor):
                attention_mask.append(am)
            else:
                attention_mask.append(torch.tensor(am))
            
            lbl = item["labels"]
            if isinstance(lbl, torch.Tensor):
                labels.append(lbl)
            else:
                labels.append(torch.tensor(lbl))
        
        # pad 到批处理中的最大长度
        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
    )
    
    # 加载数据并只取第一条
    process_fn = partial(process_sample, tokenizer=tokenizer, prompt_dict=Prompt_dict)
    train_df = pd.read_json("training_data.jsonl", lines=True)
    first_sample = train_df.iloc[0].to_dict()
    print("First sample:", first_sample)
    
    # 处理数据
    processed_sample = process_fn(first_sample)
    train_dataset = [processed_sample]
    print("Processed sample keys:", processed_sample)

First sample: {'instruction': '根据现象分析', 'input': '根据现象进行分析： QPOs in 4U 1626-67 The low-mass X-ray binary pulsar 4U 1626-67 shows quasi-periodic oscillations (QPOs) with a centroid frequency of 0.048 Hz and red noise variability as well as coherent pulsations at the 0.130 Hz neutron star rotation frequency. In power density spectra of observations made with the Rossi X-ray Timing Explorer, we have found significant sidebands at the frequencies (n*f_0 - f_qpo) and (n*f_0 + f_qpo), where f_0 = 0.130 Hz is the pulsar spin frequency, f_qpo = 0.048 Hz is the QPO frequency, and n = 1,2,3... is an integer. These sidebands provide a diagnostic of the QPO mechanism.\nIn the 17-30 keV range the powers in the sidebands are symmetric about the harmonic frequencies. This suggests that the instantaneous amplitude of the coherent pulsations is modulated by the QPOs. This phenomenon is expected (for example) in models such as the magnetospheric beat frequency model (MBFM) where the QPOs originate near 

In [37]:
model = get_peft_model(model, lora_config)
print(model)
model.print_trainable_parameters()
    
# 确保模型处于训练模式并启用梯度
model.train()
for param in model.parameters():
    param.requires_grad = True
    
args = TrainingArguments(
    report_to="none",
    output_dir="outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_steps=5,
    learning_rate=5e-4,
    lr_scheduler_type="constant",
    logging_steps=1,
    save_steps=5,
    remove_unused_columns=False,
    gradient_checkpointing=False,
    fp16=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset
)

trainer.train()

d:\anaconda\envs\d2l_env\Lib\site-packages\peft\mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
d:\anaconda\envs\d2l_env\Lib\site-packages\peft\mapping_func.py:79: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'Qwen/Qwen2.5-1.5B-Instruct' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(


PeftModel(
  (base_model): LoraModel(
    (model): PeftModel(
      (base_model): LoraModel(
        (model): Qwen2ForCausalLM(
          (model): Qwen2Model(
            (embed_tokens): Embedding(151936, 1536)
            (layers): ModuleList(
              (0-27): 28 x Qwen2DecoderLayer(
                (self_attn): Qwen2Attention(
                  (q_proj): lora.Linear(
                    (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=1536, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=1536, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
                    (lora_embedding_B): Par

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 GiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 3.66 GiB is allocated by PyTorch, and 5.63 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [40]:
messages = [
    {"role": "user", "content": "根据现象进行分析： QPOs in 4U 1626-67 The low-mass X-ray binary pulsar 4U 1626-67 shows quasi-periodic oscillations (QPOs) with a centroid frequency of 0.048 Hz and red noise variability as well as coherent pulsations at the 0.130 Hz neutron star rotation frequency. In power density spectra of observations made with the Rossi X-ray Timing Explorer, we have found significant sidebands at the frequencies (n*f_0 - f_qpo) and (n*f_0 + f_qpo), where f_0 = 0.130 Hz is the pulsar spin frequency, f_qpo = 0.048 Hz is the QPO frequency, and n = 1,2,3... is an integer. These sidebands provide a diagnostic of the QPO mechanism.\nIn the 17-30 keV range the powers in the sidebands are symmetric about the harmonic frequencies. This suggests that the instantaneous amplitude of the coherent pulsations is modulated by the QPOs. This phenomenon is expected (for example) in models such as the magnetospheric beat frequency model (MBFM) where the QPOs originate near the polar caps of the neutron star, since any variation in the X-ray emission from the polar caps will affect the intensity of the coherent pulsations.\nIn the 4-8 keV range, however, the lower-frequency sidebands (at n*f_0 - f_qpo) are significantly stronger than their higher-frequency complements (at n*f_0 + f_qpo). Since simple amplitude modulation produces side bands with equal powers, there must be an additional oscillation at the frequencies (n*f_0 - f_qpo) that produces the excess power observed in the enhanced lower-frequency sidebands. In the MBFM there is nothing obvious that would explain the enhanced lower-frequency sidebands. Thus the observed sideband structure is inconsistent with the MBFM being the explanation for the 0.048 Hz QPOs.\nA scenario that explains the 0.048 Hz QPOs as well as the observed sideband structure is the following. Suppose a coherent structure (a \"blob\" of some kind) orbits the neutron star with an orbital frequency of 0.048 Hz, which may or may not be a Keplerian frequency (e.g., it may represent a wave packet traveling in the accretion disk). This blob modulates the optical depth along the line of sight as it orbits, producing the 0.048 Hz QPOs. When it crosses the line of sight between the Earth and the neutron star, it attenuates the pulsar beam; this modulates the coherent pulsations at 0.048 Hz and produces the symmetric sidebands. The blob orbits in the same direction as the neutron star rotation, so it reprocesses the pulsar beam at the beat frequencies between the pulsar harmonics and the QPOs (n*f_0 - f_qpo). Some of the reprocessed X-rays are returned along the line of sight, producing the enhanced lower-frequency side bands.\nQuite independently, we find no evidence that the red noise variability modulates the amplitude of the coherent pulsations. This is also in contrast to the expectations of the MBFM and differs from the behavior in some high-mass X-ray binary pulsars.\nMore information can be found in our manuscript (in preparation), available as a PostScript file at the URL associated with this telegram.\nPostScript for 1626 manuscript 1997-12-17 17:28:00"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=2560, do_sample=True, temperature=0.7)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(f"Qwen: {response}\n")

Qwen: The passage describes the characteristics of a low-mass X-ray binary pulsar known as 4U 1626-67. It discusses various phenomena including quasi-periodic oscillations (QPOs) with specific properties and patterns observed through different energy ranges.

### Key Observations:

1. **Quasi-Periodic Oscillations (QPOs):**
   - The QPO has a centroid frequency of 0.048 Hz and exhibits both red noise variability and coherent pulsations at the neutron star's rotation frequency of 0.130 Hz.
   
2. **Power Density Spectra:**
   - Significant sidebands were detected at frequencies \( n \cdot f_0 - f_{qpo} \) and \( n \cdot f_0 + f_{qpo} \), where \( f_0 = 0.130 \) Hz and \( f_{qpo} = 0.048 \) Hz. These sidebands indicate the presence of coherent pulsations that are modulated by the QPOs.

3. **Sideband Symmetry:**
   - In the 17-30 keV range, the power in the sidebands was symmetric around the harmonic frequencies. This suggested that the instantaneous amplitude of the coherent pulsations 